[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Audio_Speech_DSP.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Audio & Speech DSP

The most tangible application of everything in the DSP track: sound. We synthesize, analyze, and mangle audio with the tools you already own — spectrograms, filters, and source-filter models. Every cell produces a signal you can export and *listen to* (`scipy.io.wavfile.write`; in Jupyter, `IPython.display.Audio(x, rate=fs)`).

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (STFT).
- [Filter Design](./Filter_Design.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)
fs = 16_000

def show_spec(x, title="", fs=fs):
    f, t, S = sig.stft(x, fs=fs, nperseg=512)
    plt.figure(figsize=(8.5, 2.8))
    plt.pcolormesh(t, f, 20*np.log10(np.abs(S) + 1e-8), shading="auto", vmin=-100, vmax=-20)
    plt.ylabel("Hz"); plt.xlabel("s"); plt.title(title); plt.colorbar(label="dB")
    plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *Reading Spectrograms* (~35 min)
**Goal:** learn to sight-read time–frequency pictures: tones, chirps, harmonics, percussion.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S6. &nbsp; **Feeds into:** Session 2 (speech).

---

## 2. The Spectrogram as Sheet Music

💡 **Intuition.** A spectrogram is *sheet music extracted from sound*: time runs right, pitch runs up, ink is energy. Horizontal lines = steady tones; stacked lines = harmonics of one note (their spacing IS the pitch); vertical stripes = clicks/percussion (uncertainty principle: sharp in time ⇒ smeared in frequency); sweeps = chirps. Learn these four glyphs and you can 'read' most sounds before hearing them.

In [ ]:
# three notes with harmonics (a mini 'instrument')
# percussion: two clicks

# YOUR CODE HERE


**What just happened.** Read the picture before reading this. Three groups of evenly-stacked horizontal lines are the three notes, each with four harmonics; two vertical stripes at 0.5 s and 1.3 s are the clicks. Those are two of the four glyphs, and between them they cover most of what real spectrograms contain.

**Pitch is the spacing, not the lowest line.** Each note shows lines at $f_0, 2f_0, 3f_0, 4f_0$, and the *gap* between adjacent lines equals $f_0$. This is worth insisting on because of the **missing fundamental**: filter out the 220 Hz component entirely and the note still sounds like A3, since the ear infers pitch from harmonic spacing rather than from the presence of the lowest partial. It is why a small phone speaker with no real bass response still conveys a bass line, and it is the reason Session 2's pitch estimator works by looking for periodicity rather than for a peak.

The three notes here are A3 (220 Hz), C#4 (277), E4 (330) — an A-major triad. Their harmonic stacks are identical in structure and differ only in spacing.

**The clicks are the uncertainty principle, drawn.** Each is 80 samples, about 5 ms, and each smears across the *entire* frequency axis. The notes last 0.7 s and appear as thin, well-defined lines. Perfectly localised in time means maximally spread in frequency, and vice versa — the [theorem from Foundations 1](./Foundations_of_Signal_Processing_1.ipynb), now as a picture rather than an inequality.

The striking part is that this is a property of the *analysis*, not of the signal. Change `nperseg` in `show_spec` from 512 to 128 and re-run: the clicks sharpen into crisp vertical lines while the harmonic stacks blur into an indistinct smear. Nothing about the sound changed — only the window through which we looked at it. A spectrogram is not a neutral photograph of a signal; it is one point on a resolution trade-off, and choosing the window is choosing what you are able to see. Music transcription wants a long window, drum-onset detection a short one, and systems that need both compute several in parallel.

One construction detail: each note is multiplied by a Hann envelope, so its onset is gradual. Without that, the abrupt starts would themselves produce vertical stripes — a click is simply an amplitude discontinuity, and a note that switches on instantaneously contains one.

---
### 🕐 Session 2 of 3 — *Speech: the Source-Filter Model* (~40 min)
**Goal:** synthesize vowels from scratch; estimate pitch and formants from a signal.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (effects).

---

## 3. How Speech Works

💡 **Intuition.** Speech is a two-stage instrument: the **source** (vocal cords buzzing at the pitch $f_0$, or noise for whispers/fricatives) drives a **filter** (the vocal tract, whose resonances — *formants* — are shaped by your tongue and lips). Vowels are *filter settings*: /a/ vs /i/ differ in formant positions, not pitch. This source-filter factorization is the basis of vocoders, LPC compression (your phone), and autotune.

In [ ]:
# classic formant tables: /a/ (father) vs /i/ (see)

# YOUR CODE HERE


**What just happened.** Two synthetic vowels, and the spectrogram shows exactly the factorisation the model predicts. Both halves have **identical harmonic spacing** — same pitch, 120 Hz, same source. What differs is *which* harmonics are loud: /a/ has bright bands near 730 and 1090 Hz sitting close together, while /i/ has one very low band near 270 Hz and a widely separated pair up at 2290 and 3010 Hz.

That is the whole source-filter model in one picture. The **source** sets the harmonic comb — its spacing is the pitch, and every harmonic is present. The **filter** sets the envelope — which harmonics get amplified. Vowel identity lives entirely in the envelope, so /a/ and /i/ at the same pitch differ in formants, not in their harmonic structure. Sing a steady note and morph between the two vowels: the pitch does not move, the resonances do.

The strongest evidence for this factorisation is whispering. A whisper has no pitch at all — the source is noise rather than a pulse train, so there is no comb — and speech remains perfectly intelligible. The intelligible content is in the filter; pitch carries prosody and speaker identity. This is exactly why LPC codecs work: send the filter coefficients accurately and describe the source cheaply, and you have compressed speech to a few kbit/s. Your phone call is running this model.

**And the code is doing DSP the room already knows.** Each formant is created by `sig.lfilter([1], [1, -2*r*cos(theta), r**2], x)` — a denominator with a complex-conjugate **pole pair**. The pole angle $\theta = 2\pi f_c/f_s$ sets the formant frequency, and the radius $r = e^{-\pi\,\mathrm{bw}/f_s}$ sets its bandwidth, with poles nearer the unit circle giving sharper resonances. So a formant is a pole, the vocal tract is an all-pole filter, and the [pole-zero plots from Filter Design](./Filter_Design.ipynb) are a map of anatomy. The all-pole assumption is physically motivated too — a tube with resonances has poles and few zeros — which is why LPC-based vocoders sound worst on nasals, where a side branch does introduce zeros.

The next cell inverts this: given only the waveform, recover the pitch and the formants that produced it.

In [ ]:
# Pitch estimation by autocorrelation: the lag where the signal rhymes with itself
# Formant estimation via LPC (all-pole fit — Wiener/Yule-Walker in disguise)

# YOUR CODE HERE


**What just happened.** Both halves of the source-filter model, recovered from the waveform alone.

**The pitch estimate is exact — and the printed numbers hide that.** It reports 120.3 Hz against a synthesis target of 120 Hz, which looks like 0.3 Hz of error. It is not. Look at `src[::int(fs/f0)]`: with $f_s = 16000$ and $f_0 = 120$, `int(16000/120)` truncates to **133** samples, so the signal we actually built has a period of 133 samples and a true pitch of $16000/133 = 120.3008$ Hz. The autocorrelation estimator found precisely that. The 0.3 Hz lives in the *synthesiser's* integer rounding, not in the estimate.

Worth pausing on, because the mistake is general: the ground truth is what the code produced, not what the variable was named. An error attributed to the wrong stage sends you tuning an estimator that was already perfect. Always check what your reference actually is.

The method itself is the missing-fundamental idea from Session 1 made into an algorithm. Autocorrelation asks "at what lag does this signal rhyme with itself?" — it looks for *periodicity*, not for a spectral peak, so it still works when the fundamental is weak or absent entirely.

**The formants come back close, and the errors are informative.** For /a/: [723, 1088, 2365] against [730, 1090, 2440]. For /i/: [257, 2288, 3010] against [270, 2290, 3010]. The middle formants are nearly exact (F2 of /a/ is 0.2% off, F2 of /i/ 0.1%), while the errors concentrate at the extremes — F3 of /a/ is 3.1% low and F1 of /i/ is 4.8% low.

Two mechanisms, both worth knowing. Broad formants are estimated less precisely: bandwidth 170 Hz puts the pole further from the unit circle, so the spectral peak is flatter and its location less sharply determined than a 60 Hz formant's. And LPC order 10 provides only five pole pairs to cover the entire 0–8 kHz range, so the fit spends its poles where the energy is and economises at the top, pulling F3 downward. Raise `order` to 14 and F3 improves visibly — a one-character experiment worth running.

**What `lpc_formants` is really doing.** `solve_toeplitz(r[:-1], r[1:])` is the Yule–Walker equations — the same normal equations as the Wiener filter in [Statistical SP](./Statistical_Signal_Processing.ipynb) and the same AR fit as [Classical Forecasting](../Intro_Time_Series/Classical_Forecasting.ipynb). Then `np.roots` finds the poles and their angles become frequencies. Speech coding, spectral estimation, and time-series forecasting are running identical mathematics; only the interpretation of the poles differs.

Note finally that estimation here is on *synthetic* speech that exactly obeys the all-pole model. Real speech has a glottal source with its own spectral tilt, nasal zeros, and noise — so real formant tracking is substantially harder, and the clean agreement above reflects a matched model as much as a good estimator.

---
### 🕐 Session 3 of 3 — *Effects Are Filters* (~35 min)
**Goal:** build reverb, robot voice, and a pitch shifter — and see each as a DSP primitive.
**Builds on:** Session 2.

---

## 4. The Effects Rack

Every studio effect is a signal-processing primitive wearing a costume:

| Effect | DSP primitive |
|---|---|
| Echo/reverb | convolution with a (sparse/dense) impulse response |
| Robot voice | ring modulation (multiply by a carrier) |
| Wah / EQ | time-varying / fixed [filters](./Filter_Design.ipynb) |
| Pitch shift | STFT: stretch time, then resample ([multirate](./Foundations_of_Signal_Processing_2.ipynb)) |

In [ ]:
# 1) Reverb: convolve with an exponentially decaying random impulse response
# 2) Robot: ring-modulate with a 70 Hz carrier
# 3) Pitch shift up a fourth: phase-vocoder-lite = time-stretch (STFT hop trick) + resample

# YOUR CODE HERE


**What just happened.** Three recognisable studio effects, each about four lines, and each a DSP primitive the room already owns.

**Reverb (left) is convolution, nothing more.** The impulse response is "what the room does to a clap" — here an exponentially decaying noise burst — and convolving the dry voice with it places that voice in the room. The spectrogram shows every event smeared *rightward* in time, energy trailing after each onset, with the harmonic structure preserved because convolution with a broadband IR does not move frequencies. Note `ir[0] = 1.0`: that spike preserves the direct path, and without it you would hear only reflections, as though the speaker were facing away. Commercial convolution reverbs work exactly this way, using impulse responses *measured* in real concert halls and cathedrals.

**Ring modulation (middle) is a trigonometric identity.** $\sin A\sin B = \tfrac12[\cos(A-B) - \cos(A+B)]$, so multiplying by a 70 Hz carrier splits every component into a sum-and-difference pair. The spectrogram shows the sidebands clearly — each original harmonic has become two, displaced by ±70 Hz.

The reason it sounds *inhuman* rather than merely pitch-shifted is the point worth extracting. The original partials sat at $f_0, 2f_0, 3f_0,\dots$ — a harmonic series. After modulation they sit at $nf_0 \pm 70$, and those are **no longer integer multiples of anything**. The ear identifies pitch from harmonic spacing (Session 1), so a spectrum with no consistent spacing has no coherent pitch, and we hear a metallic, robotic timbre. Incidentally, this same operation is the mixer in a superheterodyne receiver from [Digital Communications](./Digital_Communications.ipynb) — a Dalek voice and a radio front end are one circuit.

**Pitch shifting (right) is multirate processing in two moves.** First time-stretch without altering pitch, by analysing at one hop and re-synthesising at a smaller one; then resample back to the original duration, which raises the pitch. Ask why you cannot just resample directly: that changes speed and pitch together — the tape-machine effect — and it also drags the *formants* along, which is what makes the chipmunk sound. Session 2 explains why that is wrong: formants define vowel identity, so shifting them changes *who is speaking*, not just the note. Real pitch correction moves the source and leaves the filter alone.

**Be honest about this implementation.** Reusing `istft` with a mismatched `noverlap` stretches the signal but does not reconcile phase between frames, so the output has the characteristic "phasiness" of a naive phase vocoder. Proper implementations propagate phase across frames and handle transients separately. This demonstrates the mechanism; it is not a good pitch shifter.

**And the deflationary conclusion.** Reverb is convolution, robot voice is modulation, pitch shift is resampling, EQ and wah are filters. The whole effects industry is the DSP in this track, packaged and marketed. That should be encouraging rather than disappointing: you already have the tools, and the export line in this cell means you can take the results away and listen to them.

## 5. Conclusion

Spectrograms are readable sheet music; speech factors into source × filter (pitch × formants); and the entire effects industry is convolution, modulation, filtering, and resampling with good marketing. Record a real voice and rerun every cell — that's the homework.

---
## Where next

- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — classifying exactly these spectrograms.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate machinery under the pitch shifter.
- [Array Processing](./Array_Processing.ipynb) — what the *second* microphone buys you.